In [4]:
!pip install -q youtube-transcript-api langchain-community langchain-openai \
               faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [11]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [ ]:
!pip install -U youtube-transcript-api

In [ ]:
!pip install -q langchain

In [ ]:
!pip install -U langchain-text-splitters

## Step 1a - Indexing (Document Ingestion)

In [15]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound

video_id = "Gfr50f6ZBvo"

try:
    api = YouTubeTranscriptApi()
    transcript_data = api.fetch(video_id, languages=["en"])
    transcript = " ".join(chunk.text for chunk in transcript_data)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")
except NoTranscriptFound:
    print("No English transcript found for this video.")
except Exception as e:
    print(f"An error occurred: {e}")


An error occurred: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=Gfr50f6ZBvo! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

There are two things you can do to work around this:
1. Use proxies to hide your IP address, as explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).
2. (NOT RECOMMENDED) If you authenticate your requests using cookies, you will be able to continue doing requests for a while. However, YouTube will eventually permanently ban the account that you have used to 

## Step 1b -Indexing (Text Splitting)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

### Using Hugging Face Embeddings as an alternative

If you prefer to use a free, open-source model for embeddings, you can use models from Hugging Face. Many of these can be run locally without needing an API key. We'll use the `HuggingFaceEmbeddings` integration from `langchain-huggingface`.

In [12]:
!pip install -q langchain-huggingface sentence-transformers

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize a Hugging Face embedding model
# You can choose different models from the Hugging Face model hub
# This model ('all-MiniLM-L6-v2') is a good balance of performance and size
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Now, you can run the cell below this one to create the vector store with Hugging Face Embeddings

In [ ]:
pip install -U langchain-huggingface

In [ ]:
vector_store = FAISS.from_documents(chunks, embedding)
print("Vector store created successfully using Hugging Face embeddings.")

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
vector_store.get_by_ids(['3b0e7df2-e461-47e1-9fcc-2796c931458d'])

## Step 2 - Retrieval

In [ ]:
retriver = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
retriver

In [ ]:
retriver.invoke('what is deppmind')

## Step 3 - Augmentation

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
import os

# Get the Hugging Face token (assuming it's set as a Colab secret 'HF_TOKEN')
hf_token = os.getenv("HF_TOKEN")

# 1. First, create an instance of the underlying Hugging Face model using HuggingFaceEndpoint
#    This model will handle the text generation from the specified repo_id
base_llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Pro",
    temperature=0.2,
    huggingfacehub_api_token=hf_token # Pass the API token here
)

# 2. Then, wrap this base LLM with ChatHuggingFace
#    ChatHuggingFace provides a conversational interface for the base_llm
llm = ChatHuggingFace(llm=base_llm)

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
import os

In [ ]:
prompt  = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables=["context", "question"]
)


In [ ]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriver.invoke(question)

In [ ]:
retrieved_docs

In [ ]:
content_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
print(content_text)

In [ ]:
final_prompt = prompt.invoke({"context": content_text, "question": question})
print(final_prompt)

## Step 4 Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer)

## Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('who is Demis')

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')